# Sync gold → Cosmos DB (mirror-like)

Keeps the Cosmos container `ProductSummary` in step with the lakehouse table `gold_product_review_summary`. Workspace `cosmos db`, default lakehouse `retail_ai_lh`.

| Step | What it does | Why |
|---|---|---|
| 1 | Rebuild gold with **MERGE** (not overwrite) | Only rows whose values changed get a new version, so the change feed stays small |
| 2 | Read the **Change Data Feed** since the last synced version | Cost scales with changes, not table size |
| 3 | Upsert changed rows and **delete removed products** in Cosmos | Cosmos matches gold, including deletes |
| 4 | Save the bookmark, then bump `dataVersion` in `AnswerCache` | Next run starts where this one ended; cached AI answers are invalidated |
| 5 | Check counts match | The run fails loudly if Cosmos and gold disagree |

**Safe to rerun.** Upserts and deletes are idempotent, and the bookmark is saved only after every write succeeds, so a failed run is simply retried. The first run (no bookmark yet) does a full sync and removes any Cosmos docs that are no longer in gold.

Auth uses the notebook identity (`notebookutils.credentials.getToken`); no keys.

In [ ]:
# Parameters: a pipeline Notebook activity can override any of these.
FULL_RESYNC = False      # True = rewrite every doc from gold and remove orphans (use after a schema change)
COSMOS_ENDPOINT = "https://ae268e38-3ca9-4db0-815c-2c9c808d346b.zae.sql.cosmos.fabric.microsoft.com"
COSMOS_DATABASE = "retail_cosmos"
COSMOS_CONTAINER = "ProductSummary"
CACHE_CONTAINER = "AnswerCache"   # holds the dataVersion doc the API's answer cache checks; "" to skip
GOLD_TABLE = "gold_product_review_summary"
# Gold SQL generated from the KPI registry (operational-api/registry/metrics.json) and uploaded by
# `node scripts/generate-registry-sql.js --upload`. This notebook holds no metric definitions of its own.
REGISTRY_MANIFEST = "Files/registry/gold_product_review_summary.json"
STATE_TABLE = "sync_state"        # one row per target: last gold version applied
WRITE_WORKERS = 8                 # parallel Cosmos writes; the SDK retries 429s on its own

In [ ]:
# Install the Cosmos SDK on the driver if missing. Plain pip (not %pip) so this also works in
# scheduled and pipeline runs, where inline %pip is disabled by default.
import importlib.util, subprocess, sys
if importlib.util.find_spec("azure.cosmos") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "azure-cosmos>=4.14.0", "-q"], check=True)

In [ ]:
import base64, json, time, datetime, decimal, threading
import notebookutils  # built into Fabric notebooks; the explicit import also works in Livy/Spark job sessions
from concurrent.futures import ThreadPoolExecutor
from azure.core.credentials import AccessToken, TokenCredential
from azure.cosmos import CosmosClient, PartitionKey, ThroughputProperties, exceptions
from delta.tables import DeltaTable
from pyspark.sql import Window, functions as F

class FabricTokenCredential(TokenCredential):
    """Notebook identity token for Cosmos DB in Fabric."""
    def get_token(self, *scopes, **kwargs):
        tok = notebookutils.credentials.getToken("https://cosmos.azure.com/")
        payload = tok.split(".")[1]; payload += "=" * (-len(payload) % 4)
        return AccessToken(tok, json.loads(base64.urlsafe_b64decode(payload))["exp"])

db = CosmosClient(COSMOS_ENDPOINT, credential=FabricTokenCredential()).get_database_client(COSMOS_DATABASE)
container = db.create_container_if_not_exists(
    id=COSMOS_CONTAINER,
    partition_key=PartitionKey(path="/productId"),
    offer_throughput=ThroughputProperties(auto_scale_max_throughput=1000),  # Fabric requires autoscale
)
print("Cosmos ready:", COSMOS_CONTAINER)

## Step 1: rebuild gold with MERGE
The SQL comes from the **KPI registry**, the only place metrics are defined. The generated manifest carries the SQL, the business columns and the `registryVersion`/`registryHash` that are stamped on every gold row and Cosmos doc. To change a definition, edit `registry/metrics.json`, then run `node scripts/generate-registry-sql.js --upload`, and the next run of this notebook picks it up.

A row is updated only when a business column changed (`<=>` is the null-safe equals), so `refreshedAt` means "last real change" and unchanged products produce no change-feed entries. If the registry changes the gold **schema** (a metric is added, for example), gold is rewritten and Cosmos gets a full resync automatically.

In [ ]:
manifest = json.loads(notebookutils.fs.head(REGISTRY_MANIFEST, 10_000_000))
assert manifest["table"] == GOLD_TABLE, f"Manifest is for {manifest['table']}, not {GOLD_TABLE}"
assert manifest["keys"] == ["productId"], "This sync keys Cosmos docs by productId"
print(f"KPI registry v{manifest['registryVersion']} ({manifest['registryHash']})")

new_gold = spark.sql(manifest["sql"])
BUSINESS_COLS = manifest["businessColumns"]

def schema_of(df):
    return [(f.name, f.dataType.simpleString()) for f in df.schema.fields]

if not spark.catalog.tableExists(GOLD_TABLE):
    new_gold.write.format("delta").option("delta.enableChangeDataFeed", "true").saveAsTable(GOLD_TABLE)
    print("Created", GOLD_TABLE, "with Change Data Feed on")
elif schema_of(spark.table(GOLD_TABLE)) != schema_of(new_gold):
    # The registry changed the shape of gold: rewrite it and resync every Cosmos doc.
    print("Gold schema changed by the registry:\n  was", schema_of(spark.table(GOLD_TABLE)), "\n  now", schema_of(new_gold))
    (new_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .option("delta.enableChangeDataFeed", "true").saveAsTable(GOLD_TABLE))
    FULL_RESYNC = True
else:
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {GOLD_TABLE}").collect()}
    if props.get("delta.enableChangeDataFeed") != "true":
        spark.sql(f"ALTER TABLE {GOLD_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print("Enabled Change Data Feed on", GOLD_TABLE)
    changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in BUSINESS_COLS)
    (DeltaTable.forName(spark, GOLD_TABLE).alias("t")
        .merge(new_gold.alias("s"), "t.productId = s.productId")
        .whenMatchedUpdateAll(condition=changed)
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute())

last_commit = DeltaTable.forName(spark, GOLD_TABLE).history(1).collect()[0]
gold_version = last_commit["version"]
metrics = last_commit["operationMetrics"] or {}
print(f"Gold is at version {gold_version} ({last_commit['operation']}):",
      {k: v for k, v in metrics.items() if k.startswith(("numTargetRows", "numOutputRows"))})

## Step 2: work out what Cosmos is missing
- **No bookmark yet, or `FULL_RESYNC`:** full sync. Every gold row is upserted, and Cosmos docs whose product is no longer in gold are deleted.
- **Bookmark behind gold:** read the change feed from the next version. If a product changed several times, only its latest change is applied.
- **Change feed unavailable** for that range (e.g. it was enabled after the bookmark, or old versions were vacuumed): fall back to a full sync.

In [ ]:
TARGET = f"cosmos:{COSMOS_DATABASE}.{COSMOS_CONTAINER}"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {STATE_TABLE}
              (target STRING, source_table STRING, last_version BIGINT, synced_at TIMESTAMP) USING DELTA""")
state = spark.table(STATE_TABLE).where(F.col("target") == TARGET).collect()
last_version = state[0]["last_version"] if state else None

if FULL_RESYNC or last_version is None:
    mode = "full"
elif last_version >= gold_version:
    mode = "noop"
else:
    mode = "incremental"

upserts, deletes = [], []
if mode == "incremental":
    try:
        feed = (spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", last_version + 1)
                .option("endingVersion", gold_version)
                .table(GOLD_TABLE)
                .where("_change_type != 'update_preimage'"))
        latest = (feed.withColumn("_rn", F.row_number().over(
                      Window.partitionBy("productId").orderBy(F.col("_commit_version").desc())))
                  .where("_rn = 1").drop("_rn"))
        meta_cols = ["_change_type", "_commit_version", "_commit_timestamp"]
        upserts = [r.asDict() for r in latest.where("_change_type != 'delete'").drop(*meta_cols).collect()]
        deletes = [r["productId"] for r in latest.where("_change_type = 'delete'").select("productId").collect()]
    except Exception as e:
        print("Change feed unavailable for this range, doing a full sync instead:", str(e).splitlines()[0])
        mode = "full"

if mode == "full":
    upserts = [r.asDict() for r in spark.sql(f"SELECT * FROM {GOLD_TABLE} VERSION AS OF {gold_version}").collect()]
    gold_ids = {u["productId"] for u in upserts}
    cosmos_ids = [d["id"] for d in container.query_items(
        "SELECT c.id FROM c WHERE c.docType = 'productSummary'", enable_cross_partition_query=True)]
    deletes = [i for i in cosmos_ids if i not in gold_ids]

print(f"Mode: {mode}  (gold v{gold_version}, last synced v{last_version})  ->  {len(upserts)} upserts, {len(deletes)} deletes")

## Step 3: apply the changes to Cosmos
Writes run in parallel. `id = productId` (the partition key), so the app reads each product with a ~1 RU point read. Any failed write stops the run before the bookmark moves.

In [ ]:
def to_json_safe(v):
    if isinstance(v, (datetime.date, datetime.datetime)): return v.isoformat()
    if isinstance(v, decimal.Decimal): return float(v)
    return v

def to_doc(row):
    doc = {k: to_json_safe(v) for k, v in row.items()}
    doc["id"] = doc["productId"]      # id = partition key -> point reads
    doc["docType"] = "productSummary"
    return doc

ru, ru_lock = [0.0], threading.Lock()
def charge(headers, _result):
    with ru_lock:
        ru[0] += float(headers.get("x-ms-request-charge", 0))

def upsert(row):
    container.upsert_item(to_doc(row), response_hook=charge)

def delete(pid):
    try:
        container.delete_item(pid, partition_key=pid, response_hook=charge)
    except exceptions.CosmosResourceNotFoundError:
        pass  # already gone: deletes are idempotent

t0 = time.time()
with ThreadPoolExecutor(max_workers=WRITE_WORKERS) as pool:
    list(pool.map(upsert, upserts))   # list() re-raises the first failure
    list(pool.map(delete, deletes))
print(f"Applied {len(upserts)} upserts and {len(deletes)} deletes in {time.time() - t0:.1f}s, {ru[0]:.0f} RU")

## Step 4: save the bookmark and invalidate cached AI answers
The bookmark moves only now, after every write succeeded. If anything changed, a new `dataVersion` makes the API's semantic answer cache ignore answers computed from older data.

In [ ]:
(spark.createDataFrame(
    [(TARGET, GOLD_TABLE, int(gold_version), datetime.datetime.utcnow())],
    "target string, source_table string, last_version bigint, synced_at timestamp")
 .createOrReplaceTempView("new_state"))
spark.sql(f"""MERGE INTO {STATE_TABLE} t USING new_state s ON t.target = s.target
              WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
print(f"Bookmark: {TARGET} = gold v{gold_version}")

if (upserts or deletes) and CACHE_CONTAINER:
    now = datetime.datetime.now(datetime.timezone.utc).isoformat()
    _ = db.get_container_client(CACHE_CONTAINER).upsert_item(
        {"id": "dataVersion", "scope": "_system", "version": now, "updatedAt": now})
    print("Answer cache invalidated: dataVersion =", now)

## Step 5: verify
Counts must match. Fails the run (and so the pipeline) if they don't.

In [ ]:
gold_count = spark.table(GOLD_TABLE).count()
cosmos_count = list(container.query_items(
    "SELECT VALUE COUNT(1) FROM c WHERE c.docType = 'productSummary'", enable_cross_partition_query=True))[0]
print(f"Gold rows: {gold_count}   Cosmos docs: {cosmos_count}")
assert gold_count == cosmos_count, "Cosmos is out of sync with gold; rerun with FULL_RESYNC = True"

sample = spark.table(GOLD_TABLE).select("productId").first()
if sample:
    pid = sample["productId"]
    t0 = time.perf_counter()
    item = container.read_item(item=pid, partition_key=pid)
    print(f"Point read: {item['productName']} -> {item['reviewCount']} reviews, avg {item['avgStars']} "
          f"({(time.perf_counter() - t0) * 1000:.0f} ms, "
          f"{container.client_connection.last_response_headers['x-ms-request-charge']} RU)")
print("IN SYNC")

## Scheduling (option 1: chain it after gold)
Run this notebook every time the source data changes. Two ways:

- **Simple:** notebook ▸ **Run** ▸ **Schedule**, at the same cadence as the source refresh. This notebook already rebuilds gold first, so it is the whole chain.
- **Pipeline:** *Notebook* activity (this notebook) ▸ on success ▸ the review-insights step. Pass `FULL_RESYNC` as a base parameter when needed.

Notes:
- The install cell uses plain `pip`, so scheduled and pipeline runs need no extra settings. For faster starts, attach a Fabric **Environment** with `azure-cosmos` and the cell becomes a no-op.
- Tested on Fabric Spark runtime 2.0 (Spark 4.1, Python 3.13): full sync, no-change, delete and re-insert paths.
- `operational-api/jobs/precompute-insights.js` (review themes) runs outside Fabric today. Run it after this notebook when reviews changed; it also bumps `dataVersion`.
- The old one-shot loader `nb_load_product_summary_to_cosmos` is replaced by this notebook; keep it only for reference.